# 🎯 RAGScore — Audience & Purpose: Tailored QA Generation

Generate **targeted** QA pairs by specifying who the questions are for and why.

| Parameter | What it does | Examples |
|-----------|-------------|----------|
| `--audience` | Who will ask these questions? | `developers`, `customers`, `new-hires`, `auditors` |
| `--purpose` | What's the document for? | `faq`, `training`, `compliance`, `fine-tuning` |

**Requirements:** `ragscore >= 0.7.1` (feature branch), an OpenAI API key (or any supported provider)

## 1. Install from feature branch

In [ ]:
!pip install -q "ragscore[notebook] @ git+https://github.com/HZYAI/RagScore.git@feature/audience-purpose" openai numpy

import nest_asyncio
nest_asyncio.apply()
print("✅ Ready")

## 2. API Key

In [ ]:
import os

# Option 1: Colab Secrets (recommended — click 🔑 in the left sidebar)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Option 2: Paste directly (not recommended for shared notebooks)
    os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your key
    print("⚠️  Using hardcoded key — consider using Colab Secrets instead")

## 3. Create a Sample Document

We'll use a technical document that could serve multiple audiences.

In [ ]:
%%writefile product_docs.txt
DataSync Pro is an enterprise data integration platform that enables real-time synchronization between databases, data warehouses, and cloud storage systems. Version 3.2 was released in January 2026 with support for over 150 connectors.

Architecture Overview: DataSync Pro uses a distributed microservices architecture built on Kubernetes. The core components include the Sync Engine (handles data transformation and routing), the Connector Hub (manages source/target connections), the Schema Registry (tracks schema evolution), and the Monitoring Dashboard (real-time pipeline health). All inter-service communication uses gRPC with Protocol Buffers for type-safe, high-performance messaging.

Installation: DataSync Pro can be deployed via Helm chart (helm install datasync-pro datasync/pro --namespace datasync) or Docker Compose for development environments. Minimum requirements are 8GB RAM, 4 CPU cores, and 50GB storage. The platform requires PostgreSQL 14+ for metadata storage and Redis 7+ for caching. For production deployments, we recommend Kubernetes 1.27+ with at least 3 worker nodes.

API Reference: The REST API is available at /api/v3/. Authentication uses OAuth 2.0 with JWT tokens. Key endpoints include POST /api/v3/pipelines (create pipeline), GET /api/v3/pipelines/{id}/status (check status), POST /api/v3/pipelines/{id}/trigger (manual trigger), and DELETE /api/v3/pipelines/{id} (remove pipeline). Rate limiting is set to 1000 requests per minute per API key. The Python SDK (pip install datasync-pro-sdk) provides a higher-level interface with automatic retry and pagination.

Data Transformation: DataSync Pro supports SQL-based transformations, Python UDFs, and a visual drag-and-drop transformation builder. Transformations run in isolated containers with configurable resource limits. The platform supports Change Data Capture (CDC) for MySQL, PostgreSQL, MongoDB, and SQL Server, enabling real-time incremental sync with sub-second latency.

Security and Compliance: All data in transit is encrypted using TLS 1.3. Data at rest is encrypted with AES-256. The platform is SOC 2 Type II certified and GDPR compliant. Row-level access controls can be configured per pipeline. Audit logs are retained for 90 days by default (configurable up to 7 years for compliance). PII detection and masking is available as an add-on module.

Pricing: DataSync Pro offers three tiers. Starter ($499/month) includes up to 10 pipelines, 5 connectors, and community support. Professional ($1,499/month) includes unlimited pipelines, 50 connectors, CDC support, and email support with 4-hour SLA. Enterprise (custom pricing) includes all features, unlimited connectors, dedicated support engineer, custom SLA, and on-premise deployment option. All plans include a 14-day free trial.

Troubleshooting: Common issues include connection timeouts (increase timeout_seconds in connector config), schema drift errors (enable auto_schema_evolution in pipeline settings), and memory pressure (increase container memory limits or reduce batch_size). For CDC pipelines, ensure the source database has binary logging enabled (MySQL) or logical replication configured (PostgreSQL). Contact support@datasyncpro.com for production issues.

## 4. Build a Mini RAG

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()

# Load and chunk by paragraph
with open("product_docs.txt") as f:
    text = f.read()
chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]
print(f"✅ {len(chunks)} chunks created")

# Embed all chunks
def get_embedding(text):
    return client.embeddings.create(input=text, model="text-embedding-3-small").data[0].embedding

chunk_embeddings = np.array([get_embedding(c) for c in chunks])
print(f"✅ {len(chunk_embeddings)} chunks embedded")

# Define RAG function
def my_rag(question):
    """Embed question -> find top-3 chunks -> ask GPT-4o."""
    q_emb = np.array(get_embedding(question))
    sims = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    top_idx = np.argsort(sims)[-3:][::-1]
    context = "\n\n".join([chunks[i] for i in top_idx])

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Answer based only on the provided context. Be concise and accurate."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

# Sanity check
print("\n🧪 Sanity check:")
print(my_rag("What is DataSync Pro?"))

## 5. Baseline — No audience/purpose

First, let's see what generic QA generation looks like.

In [ ]:
from ragscore import quick_test

baseline = quick_test(my_rag, docs="product_docs.txt", n=5)
print("\n📋 Baseline questions generated:")
for _, row in baseline.df.iterrows():
    print(f"  Q: {row['question'][:100]}")

## 6. 🎯 Audience: Developers

Questions a **developer** would actually ask — API endpoints, config, troubleshooting.

In [ ]:
dev_result = quick_test(
    my_rag, docs="product_docs.txt", n=5,
    audience="developers",
    purpose="api-integration",
)
print("\n👨‍💻 Developer questions:")
for _, row in dev_result.df.iterrows():
    print(f"  Q: {row['question'][:100]}")

## 7. 🎯 Audience: Customers (Pre-sales)

Questions a **potential customer** would ask — pricing, features, compliance.

In [ ]:
customer_result = quick_test(
    my_rag, docs="product_docs.txt", n=5,
    audience="potential customers evaluating the product",
    purpose="pre-sales FAQ",
)
print("\n🛒 Customer questions:")
for _, row in customer_result.df.iterrows():
    print(f"  Q: {row['question'][:100]}")

## 8. 🎯 Audience: Compliance Auditors

Questions an **auditor** would ask — security, certifications, data handling.

In [ ]:
audit_result = quick_test(
    my_rag, docs="product_docs.txt", n=5,
    audience="compliance auditors",
    purpose="security audit",
)
print("\n🔒 Auditor questions:")
for _, row in audit_result.df.iterrows():
    print(f"  Q: {row['question'][:100]}")

## 9. Compare Results Across Audiences

Same document, different audiences — see how the questions differ.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Audience": ["Baseline", "Developers", "Customers", "Auditors"],
    "Accuracy": [
        f"{baseline.accuracy:.0%}",
        f"{dev_result.accuracy:.0%}",
        f"{customer_result.accuracy:.0%}",
        f"{audit_result.accuracy:.0%}",
    ],
    "Avg Score": [
        f"{baseline.avg_score:.1f}/5",
        f"{dev_result.avg_score:.1f}/5",
        f"{customer_result.avg_score:.1f}/5",
        f"{audit_result.avg_score:.1f}/5",
    ],
    "Sample Question": [
        baseline.df.iloc[0]["question"][:80] + "..." if len(baseline.df) > 0 else "N/A",
        dev_result.df.iloc[0]["question"][:80] + "..." if len(dev_result.df) > 0 else "N/A",
        customer_result.df.iloc[0]["question"][:80] + "..." if len(customer_result.df) > 0 else "N/A",
        audit_result.df.iloc[0]["question"][:80] + "..." if len(audit_result.df) > 0 else "N/A",
    ],
})
display(comparison)

## 10. 🧪 Synthetic Data Generation Use Case

Use `audience` + `purpose` to generate **training data** for fine-tuning.

In [ ]:
from ragscore.pipeline import run_pipeline

# Generate fine-tuning data targeted at support engineers
run_pipeline(
    paths=["product_docs.txt"],
    audience="support engineers handling customer tickets",
    purpose="fine-tuning a support chatbot",
)

# View generated QA pairs
import json
with open("output/generated_qas.jsonl") as f:
    qas = [json.loads(line) for line in f]

print(f"\n✅ Generated {len(qas)} QA pairs for fine-tuning\n")
for qa in qas[:5]:
    print(f"Q: {qa['question'][:100]}")
    print(f"A: {qa['answer'][:100]}")
    print()

## 11. CLI Usage

The same flags work from the command line:

In [ ]:
# Developer FAQ
!ragscore generate product_docs.txt --audience developers --purpose faq

# Customer onboarding material
# !ragscore generate product_docs.txt --audience "new customers" --purpose onboarding

# Compliance training data
# !ragscore generate product_docs.txt --audience auditors --purpose compliance

---

## 📚 Resources

- **GitHub**: https://github.com/HZYAI/RagScore
- **PyPI**: https://pypi.org/project/ragscore/
- **Website**: https://ragscore.io

⭐ Star us on GitHub if you find this useful!